# Космос — конец Вселенной, версия 2

Полный фильм: 26:00, Full HD 1920×1080, 25 кадров/с. Движущиеся космические кадры и 3D-визуализации; короткие немецкие подписи; без плоских схем и верхних технических блоков.

Этот ноутбук **действительно собирает видео и сохраняет результат на ваш Google Drive**. Он не является ссылкой на уже загруженный в Drive MP4. Для выполнения нужны ваша Google-сессия и исходная озвучка `l966a-enchz (1).mp3`. Ассистент не получает доступ к паролям или токенам. Старые выпуски в репозитории не меняются.

Запускайте ячейки сверху вниз. Исходники скачиваются с официальных сайтов; требуется несколько гигабайт свободного места. Рендер может занять заметное время. GPU не обязателен.

In [ ]:
import pathlib, subprocess, shutil, json
ROOT = pathlib.Path('/content/cosmos-universum-v2')
REPO = ROOT / 'repo'
ROOT.mkdir(exist_ok=True)
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch','cosmos-universum-v2','https://github.com/anzelina878-collab/v1.git',str(REPO)], check=True)
    subprocess.run(['git','-C',str(REPO),'sparse-checkout','set','cosmos'], check=True)
PROJECT = REPO / 'cosmos' / '001-universum-v2'
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg','fonts-dejavu-core'],check=True)
print(PROJECT)


## Подключить ваш диск и выбрать папку результата

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DESTINATION = pathlib.Path('/content/drive/MyDrive/Cosmos/001-universum-v2')
DESTINATION.mkdir(parents=True, exist_ok=True)
print('Готовый MP4 будет сохранён в:', DESTINATION)


## Загрузить исходную озвучку
Выберите тот же сжатый `l966a-enchz (1).mp3`, который присылали в чат. Проверяется длительность записи. Файл остаётся в вашей сессии Colab; озвучка не генерируется заново и не ускоряется.

In [ ]:
from google.colab import files
import hashlib
AUDIO = ROOT / 'voiceover.mp3'
if not AUDIO.exists():
    uploaded = files.upload()
    candidates = [(name,data) for name,data in uploaded.items() if name.lower().endswith('.mp3')]
    if len(candidates) != 1:
        raise ValueError('Выберите один исходный l966a-enchz (1).mp3.')
    AUDIO.write_bytes(candidates[0][1])
measured = float(subprocess.check_output(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',str(AUDIO)]))
if abs(measured - 1560) > 0.2:
    raise ValueError('Ожидается исходная озвучка длительностью 26:00.')
print('Озвучка загружена. Длительность:', measured)


## Скачать выбранные исходники и собрать полный фильм

In [ ]:
import sys
subprocess.run([sys.executable, str(PROJECT/'download_project_assets.py'), '--project', str(PROJECT)], check=True)
FINAL = DESTINATION / 'Universum_V2_1080p.mp4'
LOCAL_FINAL = ROOT / 'Universum_V2_1080p.mp4'
subprocess.run([sys.executable, str(PROJECT/'render.py'), '--project', str(PROJECT), '--audio', str(AUDIO), '--output', str(LOCAL_FINAL), '--jobs', '2'], check=True)
temporary = DESTINATION / (FINAL.name + '.uploading')
shutil.copyfile(LOCAL_FINAL, temporary)
if temporary.stat().st_size != LOCAL_FINAL.stat().st_size:
    raise IOError('Не удалось полностью сохранить фильм на Google Drive.')
temporary.replace(FINAL)
for name in ['Universum_DE.srt','Kapitel.txt','SOURCES.md','SKILL.md']:
    shutil.copy2(PROJECT/name, DESTINATION/name)
print('Полный фильм сохранён:', FINAL)
print('Размер:', round(FINAL.stat().st_size / 1024**2), 'МБ')
print('SHA-256:', hashlib.file_digest(FINAL.open('rb'), 'sha256').hexdigest())


## Скачать готовый MP4 на компьютер
Этот шаг необязателен: полный фильм уже находится в указанной папке Google Drive после успешной предыдущей ячейки.

In [ ]:
files.download(str(LOCAL_FINAL))
